In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from datetime import datetime
import os
import pandas as pd
import xarray as xr

import rasterio
from rasterio.transform import rowcol
from rasterio.warp import calculate_default_transform, reproject, Resampling
from netCDF4 import Dataset

In [ ]:
# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

model_input_dir = Path('/export/lv9/projects/dws/model_input/')
dws_porosity_tif = model_input_dir / 'porosity' / 'DWS_Porosity_Wu_adjusted.tif'
ns_porosity_nc = model_input_dir / 'porosity' / 'NS_Ben_Sedprop.nc'
bathymetry = model_input_dir / 'bathymetry' / 'topo_dws_500m.nc'

output_file = model_input_dir / 'porosity' / 'Ben_Sedprop.nc'

constant_ns_value = 0.401145

In [ ]:
# NS-first merge: base from NS, then overwrite with DWS where available
from rasterio.warp import transform as rio_transform


def _pick_var(ds, preferred, dims=None):
    for name in preferred:
        if name in ds.data_vars:
            if dims is None or tuple(ds[name].dims) == tuple(dims):
                return name
    for name, da in ds.data_vars.items():
        if dims is None or tuple(da.dims) == tuple(dims):
            return name
    raise ValueError('No suitable variable found')

# Bathymetry/grid
bathy_ds = xr.open_dataset(bathymetry)
if 'lonc' in bathy_ds and 'latc' in bathy_ds:
    lon2d = bathy_ds['lonc'].values.astype(np.float32)
    lat2d = bathy_ds['latc'].values.astype(np.float32)
else:
    xc_ = bathy_ds['xc'].values
    yc_ = bathy_ds['yc'].values
    lon2d, lat2d = np.meshgrid(xc_, yc_)
    lon2d = lon2d.astype(np.float32)
    lat2d = lat2d.astype(np.float32)

xc_1d = bathy_ds['xc'].values.astype(np.float32) if 'xc' in bathy_ds.coords else np.arange(lon2d.shape[1], dtype=np.float32)
yc_1d = bathy_ds['yc'].values.astype(np.float32) if 'yc' in bathy_ds.coords else np.arange(lon2d.shape[0], dtype=np.float32)

bathy_var = _pick_var(
    bathy_ds,
    preferred=['Bathymetry', 'bathymetry', 'topo', 'depth', 'elevation', 'z'],
    dims=('yc', 'xc'),
)
bathy_val = bathy_ds[bathy_var].values.astype(np.float32)
sea_mask = np.isfinite(bathy_val) & (bathy_val > -10.0)

ny, nx = sea_mask.shape
porosity = np.full((ny, nx), np.nan, dtype=np.float32)

# Source 1: NS netcdf (base field)
ns_ds = xr.open_dataset(ns_porosity_nc)
ns_var = _pick_var(ns_ds, preferred=['Porosity', 'porosity'])
ns_data = ns_ds[ns_var]
ns_val = ns_data.values.astype(np.float32)
while ns_val.ndim > 2:
    ns_val = ns_val[0, ...]

if ns_val.shape == porosity.shape:
    base = np.isfinite(ns_val) & sea_mask
    porosity[base] = ns_val[base]
elif {'lon', 'lat'}.issubset(set(ns_data.coords)):
    target_lon = xr.DataArray(lon2d, dims=('yc', 'xc'))
    target_lat = xr.DataArray(lat2d, dims=('yc', 'xc'))
    ns_lin = ns_data.interp(lon=target_lon, lat=target_lat, method='linear')
    ns_near = ns_data.interp(lon=target_lon, lat=target_lat, method='nearest')
    ns_interp = ns_lin.fillna(ns_near).values.astype(np.float32)
    base = np.isfinite(ns_interp) & sea_mask
    porosity[base] = ns_interp[base]
else:
    raise ValueError('NS file grid not alignable and has no lon/lat coords for interpolation')

# Source 2: DWS tif (overwrite where available and in sea)
with rasterio.open(dws_porosity_tif) as src:
    src_crs = src.crs
    if src_crs is None:
        raise ValueError('DWS tiff has no CRS defined.')

    if str(src_crs).lower() in ('epsg:4326', 'ogc:crs84'):
        xs, ys = lon2d.ravel(), lat2d.ravel()
    else:
        xs, ys = rio_transform('EPSG:4326', src_crs, lon2d.ravel().tolist(), lat2d.ravel().tolist())
        xs = np.asarray(xs)
        ys = np.asarray(ys)

    sampled = np.array([v[0] for v in src.sample(np.column_stack([xs, ys]))], dtype=np.float32).reshape((ny, nx))
    nodata = src.nodata
    if nodata is not None:
        sampled[np.isclose(sampled, nodata)] = np.nan

overwrite = sea_mask & np.isfinite(sampled)
porosity[overwrite] = sampled[overwrite]

# Fill any remaining sea gaps by local neighbour averaging
missing = sea_mask & ~np.isfinite(porosity)
for _ in range(600):
    if not np.any(missing):
        break
    padded = np.pad(porosity, 1, mode='constant', constant_values=np.nan)
    neigh = np.stack([
        padded[0:-2, 0:-2], padded[0:-2, 1:-1], padded[0:-2, 2:],
        padded[1:-1, 0:-2],                      padded[1:-1, 2:],
        padded[2:,   0:-2], padded[2:,   1:-1], padded[2:,   2:],
    ], axis=0)
    neigh_mean = np.nanmean(neigh, axis=0)
    can_fill = missing & np.isfinite(neigh_mean)
    if not np.any(can_fill):
        break
    porosity[can_fill] = neigh_mean[can_fill].astype(np.float32)
    missing = sea_mask & ~np.isfinite(porosity)

porosity[~sea_mask] = np.nan

out = xr.Dataset(
    data_vars={
        'Porosity': (('yc', 'xc'), porosity.astype(np.float32)),
        'lonc': (('yc', 'xc'), lon2d.astype(np.float32)),
        'latc': (('yc', 'xc'), lat2d.astype(np.float32)),
    },
    coords={
        'xc': (('xc',), xc_1d.astype(np.float32)),
        'yc': (('yc',), yc_1d.astype(np.float32)),
    },
    attrs={
        'title': 'Merged sediment porosity (North Sea + Wadden Sea replacement)',
        'source': 'Python translation of Sedprop workflow',
        'type': 'Merged Wadden Sea + North Sea porosity',
        'gridid': 'North Sea and Wadden Sea',
    },
)

encoding = {
    'Porosity': {'dtype': 'float32', '_FillValue': np.float32(np.nan)},
    'lonc': {'dtype': 'float32', '_FillValue': np.float32(-9999.0)},
    'latc': {'dtype': 'float32', '_FillValue': np.float32(-9999.0)},
    'xc': {'dtype': 'float32', '_FillValue': np.float32(-9999.0)},
    'yc': {'dtype': 'float32', '_FillValue': np.float32(-9999.0)},
}

# Write to temp then atomically replace to avoid file-lock issues
tmp_out = output_file.with_name(output_file.stem + '_tmp.nc')
out.to_netcdf(tmp_out, mode='w', format='NETCDF4_CLASSIC', encoding=encoding)
os.replace(tmp_out, output_file)

print(f'Wrote: {output_file}')
print(f'NS base coverage in sea: {int(np.sum(sea_mask & np.isfinite(ns_val if ns_val.shape==porosity.shape else np.nan)))}')
print(f'DWS overwrite points in sea: {int(np.sum(overwrite))}')
print(f'Final missing sea points: {int(np.sum(sea_mask & ~np.isfinite(porosity)))}')

In [ ]:
import subprocess, shlex
cmd = f"ncdump -h {shlex.quote(str(output_file))}"
print(subprocess.check_output(cmd, shell=True, text=True))